In [1]:
from itertools import product
from collections import Counter
import os
import re
import pickle as pkl

import nltk
from nltk.corpus import stopwords
import nltk.stem as st
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression#, SGDClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from prepare_text_class import TextPrepareClass
from utilities import seedEverything

In [2]:
rand_seed = 141592
seedEverything(rand_seed)

## Задаю переменные

In [3]:
PATH_DATA = os.path.join('.', 'data')
PATH_SUBM = os.path.join('.', 'submissions')
PATH_MODL = os.path.join('.', 'models')

In [4]:
PHONES = r'\b(samsung|galaxy|xiaomi|iphon|redmi|note|honor|huawei|apple|' +\
          'nokia|meizu|google|самсунг|айфон|mi|lenovo|lg|redme|asus|vivo|' +\
          'zte|helio|mediatek|oppo|htc|pixel|xperia|fly|realme|zenfone|' +\
          'alcatel|blade|philips|touch|lumia|oneplus|motorola|inoi|red|neo|' +\
          'moto|panasonic|band|honnor|bbk|vertex|lafleur|xiomi|редми|хонор|' +\
          'ноки|хуаве|мейзу|асус|галакси|иной|гэлакси|хонор)(|[a-zа-я]+)'

## Загрузка, очистка и преобразование данных

In [5]:
df = pd.read_csv(os.path.join(PATH_DATA, 'ru_train.csv'), )
df.shape

(4850, 5)

In [6]:
%%time
textprepare = TextPrepareClass(PHONES, stopwords.words('russian'))
df['text_cl'] = df.review.map(textprepare.clean_all)
df['target']  = df.rating.apply(lambda x: 0 if int(x) >= 4 else 1)

df = df.sample(frac=1).reset_index(drop=True)
df.sample(3)[['review', 'rating', 'link', 'target', 'text_cl']]

CPU times: user 1min, sys: 167 ms, total: 1min
Wall time: 1min


,review,rating,link,target,text_cl
3389,"Не зря Apple считают одной из лучших фирм, пот...",5.0,https://irecommend.ru/content/moya-prelest-282,0,не зря счита одн лучш фирм техник действительн...
1668,Купила это телефон примерно год назад за 5000 ...,1.0,https://irecommend.ru/content/polnyi-uzhas-foto,1,куп эт телефон примерн год назад рубл снача не...
1883,Вся моя семья фанаты сотовых телефонов фирмы s...,5.0,https://irecommend.ru/content/moya-lyubov-k-sm...,0,вся сем фанат сотов телефон фирм одн врем пере...


In [7]:
df['target'].value_counts(normalize=True)

target
0    0.548454
1    0.451546
Name: proportion, dtype: float64

## Создание и сохранение модели и токенайзера для инференса через   
## onnx + docker + flask

Очищенные отзывы векторизую через tf-idf.   
Полученные векторы в LogReg для подбора параметров.

In [8]:
grid_pipe = Pipeline([('vector', TfidfVectorizer(analyzer = 'char_wb')),
                     ('model', LogisticRegression(solver = 'liblinear',))
                     ])

params_pipe = {'vector__max_features': [None],        # 768, 1024
               'vector__ngram_range': [(2, 3), (3, 3), (3, 4), (2, 4)], # (4, 4)
               'vector__min_df': [0.1, 0.2, 0.3],     # 0.5
               'vector__max_df': [0.75, 0.85, 0.95],
               'model__penalty': ['l1'],              # 'l2'
               'model__class_weight': ['balanced'],
               'model__max_iter': [100],              # 75, 150],
               'model__C': [4, 10, 30, 100],
               }

In [9]:
%%time
clf = GridSearchCV(grid_pipe, params_pipe,
                   n_jobs=-1, cv=5,
                   scoring = ['roc_auc', 'accuracy'],
                   refit='roc_auc',
                   verbose=0,
                  )
clf.fit(df.text_cl, df.target)
clf.best_score_, clf.best_params_

CPU times: user 12.9 s, sys: 3.29 s, total: 16.2 s
Wall time: 27min 11s


(np.float64(0.9324724482438974),
 {'model__C': 4,
  'model__class_weight': 'balanced',
  'model__max_iter': 100,
  'model__penalty': 'l1',
  'vector__max_df': 0.85,
  'vector__max_features': None,
  'vector__min_df': 0.1,
  'vector__ngram_range': (3, 4)})

 Сохраняю результаты на всех возможных комбинациях заданных параметров   
 для дальнейшего анализа

In [10]:
res = pd.DataFrame(clf.cv_results_['params'])
res['mean_test_roc_auc'] = clf.cv_results_['mean_test_roc_auc']
res['mean_test_accuracy'] = clf.cv_results_['mean_test_accuracy']
res['mean_sum'] = (clf.cv_results_['mean_test_accuracy'] +\
                   clf.cv_results_['mean_test_roc_auc']
                   )/2

res.to_csv(os.path.join(PATH_DATA, 'reslt_grid1_logreg.csv'), index=False)

## Обучаю модель на лучших параметрах и полных данных

In [30]:
%%time
vectorizer = TfidfVectorizer(analyzer = 'char_wb',
                             ngram_range=clf.best_params_['vector__ngram_range'],
                             max_df=clf.best_params_['vector__max_df'],
                             min_df=clf.best_params_['vector__min_df'],
                             max_features=clf.best_params_['vector__max_features'],
                            )
vectorizer.fit(df.text_cl)
train = vectorizer.transform(df.text_cl)

model = LogisticRegression(penalty=clf.best_params_['model__penalty'],
                           solver='liblinear',
                           C=clf.best_params_['model__C'],
                           class_weight=clf.best_params_['model__class_weight'],
                           max_iter=clf.best_params_['model__max_iter'],
                          )
model.fit(train, df.target)

CPU times: user 8.91 s, sys: 131 ms, total: 9.04 s
Wall time: 9.05 s


LogisticRegression(C=4, class_weight='balanced', penalty='l1',
                   solver='liblinear')

In [31]:
zero_features = sum(model.coef_[0] == 0)
print(f'selected {train.shape[1] - zero_features} '
      + f'from {train.shape[1]} features')

selected 594 from 2929 features


In [32]:
# train.shape[1], zero_features

## Сохраняю

С учетом предполагаемого применения, будет необходимо отслеживать версии    
скриптов очистки, приведения к начальной форме и модели. Для облегчения   
этолй задачи объеденим все в класс. При изменении версий данный класс не   
будет занимать много места, об этом не беспокоюсь.    

In [33]:
with open(os.path.join(PATH_MODL, 'logreg_model.pkl'), 'wb') as fd:
    pkl.dump(model, fd)
    
with open(os.path.join(PATH_MODL, 'logreg_vektorizer.pkl'), 'wb') as fd:
    pkl.dump(vectorizer, fd)

## Посмотрим на результат обучения на полном датасете   
## с лучшими (по cv ) параметрами

In [34]:
pred_train_logreg = model.predict(train)
print( f'roc_auc_score: {roc_auc_score(df.target, pred_train_logreg)}\n'\
       f'accuracy_score: {accuracy_score(df.target, pred_train_logreg)}')

roc_auc_score: 0.9298640436708209
accuracy_score: 0.9301030927835051


In [35]:
cm = confusion_matrix(df.target, pred_train_logreg)
cm

array([[2480,  180],
       [ 159, 2031]])

## Ошибки в предсказаниях тональности отзывов

In [36]:
df['pred_train_logreg'] = pred_train_logreg
error_reviews = df[df['target'] != df['pred_train_logreg']]

In [37]:
tmp = error_reviews.sample()
print(f'True class {tmp.target.item()},\nPredicted class {tmp.pred_train_logreg.item()}\n')
print(tmp.review.item())

True class 1,
Predicted class 0

Прежде чем его приобрести, я узнала из отзывов, что у Сони в подобных моделях часто вылезает какой - то глюк. Но я пошла на риск, а зря.
 Сразу же начались проблемы со звуком.
 У меня уже долгие годы телефоны Сони, всегда была довольна. не знала проблем. Но не в этот раз.
 При обычном разговоре по телефону звук отличный, а вот при общении в вотсап, обычный звук, то пропадает, то появляется. Звук всегда есть, только если приложить трубку к уху. Также дело обстоит и со звуком в видео.
 После того, как система обновилась до следующей версии андроида, перестала работать программа для фото и видео. Скачала другую программу в плей маркет, но через нее нет возможности делать селфи.
 Аккумулятора хватает менее чем на день. Это раздражает.
 Мне нравится, что телефон солидных размеров. Все устраивает. Некоторым его неудобно держать в руках за счет его параметров. Для больших рук и сумок. В карман джинсов не поместится.
Отлично смотреть видео.
 У телефона прочное 